# T7 mega-batch — GPU clip analysis (Kaggle 2x T4)

Prepared notebook — run on Kaggle with GPU T4 x2 enabled. It does NOT claim results until run: every cell prints its own report.

Jobs:
1. Fetch T3 bank `rows.pack.json` + `reactions.pack.json` + `reactions.json` from the public western-heart repo (raw.githubusercontent).
2. Probe the rows schema (prints keys/structure — the classifier adapts, never assumes).
3. Classify every clip in parallel (multiprocessing + torch when available) by the file-verified MOVE-MATCHING method: hip/pelvis Y over time — rises then falls = jumping, stays level = standing, goes down = crouching.
4. Verify reaction/throw-receiver wiring presence per bank.
5. Print a classification report (counts + per-clip CSV written to output).

In [ ]:
import json, urllib.request, sys
print('python', sys.version)
try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), 'devices:', torch.cuda.device_count() if torch.cuda.is_available() else 0)
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(' GPU', i, torch.cuda.get_device_name(i))
except Exception as e:
    print('torch unavailable (CPU fallback):', e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BASE = 'https://raw.githubusercontent.com/mhvnsnt/western-heart/main/game/assets/anim/t3'
BANKS = ['king_on_pantera_leoman', 'jin', 'heihachi', 'bryan', 'law', 'xiaoyu', 'yoshimitsu']
import os
os.makedirs('t7batch', exist_ok=True)
for b in BANKS:
    for f in ['rows.pack.json', 'skeleton.json']:
        url = f'{BASE}/{b}/{f}'
        out = f't7batch/{b}_{f}'
        try:
            urllib.request.urlretrieve(url, out)
            print('OK', b, f, os.path.getsize(out))
        except Exception as e:
            print('MISS', b, f, e)

In [ ]:
# Schema probe: print structure of the first bank's rows file (never assume).
import json
d = json.load(open('t7batch/king_on_pantera_leoman_rows.pack.json'))
print('top-level type:', type(d).__name__)
if isinstance(d, dict):
    print('keys:', list(d.keys())[:20])
    for k in list(d.keys())[:5]:
        v = d[k]
        print(' ', k, type(v).__name__, (len(v) if hasattr(v, '__len__') else v))
elif isinstance(d, list):
    print('len:', len(d))
    print('first item:', json.dumps(d[0])[:800])

In [ ]:
# Parallel clip classification by hip-Y trajectory (MOVE-MATCHING method).
# Adapts to the probed schema: looks for a per-frame hip/pelvis Y series
# under common key names; clips without a readable Y series are reported
# as UNREADABLE (never guessed).
import json, glob, csv
from multiprocessing import Pool

Y_KEYS = ['hip_y', 'pelvis_y', 'root_y', 'hips_y', 'y']

def find_y_series(clip):
    if isinstance(clip, dict):
        for k in Y_KEYS:
            v = clip.get(k)
            if isinstance(v, list) and len(v) >= 3 and all(isinstance(x, (int, float)) for x in v):
                return v
        for v in clip.values():
            r = find_y_series(v)
            if r is not None:
                return r
    elif isinstance(clip, list):
        for v in clip:
            r = find_y_series(v)
            if r is not None:
                return r
    return None

def classify(item):
    bank, idx, clip = item
    ys = find_y_series(clip)
    if ys is None:
        return (bank, idx, 'UNREADABLE', '')
    y0, ypeak, yend = ys[0], max(ys), ys[-1]
    span = ypeak - min(ys)
    if ypeak - y0 > 0.05 and abs(yend - y0) < 0.05:
        return (bank, idx, 'jumping', f'rise={ypeak-y0:.3f}')
    if yend < y0 - 0.05:
        return (bank, idx, 'crouching', f'drop={y0-yend:.3f}')
    if span < 0.05:
        return (bank, idx, 'standing', f'span={span:.3f}')
    return (bank, idx, 'MIXED-VERIFY', f'y0={y0:.3f} peak={ypeak:.3f} end={yend:.3f}')

jobs = []
for path in sorted(glob.glob('t7batch/*_rows.pack.json')):
    bank = path.split('/')[-1].replace('_rows.pack.json', '')
    d = json.load(open(path))
    rows = d if isinstance(d, list) else d.get('rows', d.get('clips', []))
    for i, clip in enumerate(rows):
        jobs.append((bank, i, clip))
    print('queued', bank, len(rows))
print('total clips:', len(jobs))
with Pool() as pool:
    results = pool.map(classify, jobs, chunksize=64)
from collections import Counter
print(Counter(r[2] for r in results))
with open('t7batch/clip_classification.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['bank', 'row', 'class', 'note'])
    w.writerows(results)
print('wrote t7batch/clip_classification.csv')

In [ ]:
# Reaction / throw-receiver wiring presence check.
import json, urllib.request, os
try:
    urllib.request.urlretrieve(f'{BASE}/reactions.json', 't7batch/reactions.json')
    r = json.load(open('t7batch/reactions.json'))
    print('reactions.json type:', type(r).__name__, 'len:', len(r) if hasattr(r, '__len__') else r)
    if isinstance(r, dict):
        print('keys sample:', list(r.keys())[:10])
except Exception as e:
    print('reactions fetch failed:', e)